# Stacking Ensemble for Sentiment Classification
This notebook demonstrates a stacking ensemble approach for sentiment classification using Reddit comments. It combines LightGBM and Logistic Regression as base learners, with K-Nearest Neighbors as the meta-learner. The workflow includes data loading, preprocessing, feature extraction, model training, and evaluation.

### Model Training and Evaluation
Fit the stacking ensemble on the training data, make predictions on the test set, and evaluate performance using a classification report. This provides precision, recall, f1-score, and support for each sentiment class.

### Stacking Ensemble Model Setup
Combine multiple classifiers to improve predictive performance:
- **Base learners**: LightGBM and Logistic Regression learn from the TF-IDF features.
- **Meta-learner**: K-Nearest Neighbors aggregates predictions from base learners to make final decisions.
This ensemble approach leverages the strengths of each model for robust sentiment classification.

### Feature Extraction: TF-IDF Vectorization
Transform the cleaned text data into numerical features using TF-IDF with trigrams (1-3 word sequences) and a maximum of 10,000 features. This helps capture richer context from the comments for model training.

## Workflow Overview
1. **Import Libraries**: Load required Python libraries for data manipulation, feature extraction, modeling, and evaluation.
2. **Load Dataset**: Read preprocessed Reddit comments from CSV.
3. **Preprocessing**: Remove rows with missing values in the text column.
4. **Feature Extraction**: Use TF-IDF vectorization with trigrams to convert text to numerical features.
5. **Model Setup**: Define LightGBM and Logistic Regression as base learners, and KNN as the meta-learner for stacking.
6. **Training**: Fit the stacking ensemble on the training data.
7. **Evaluation**: Predict on the test set and print classification metrics.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report

# Load the dataset
dataset = pd.read_csv('reddit_preprocessing.csv')

# Drop rows with NaN values in 'clean_comment'
cleaned_dataset = dataset.dropna()

# Separate features and target
X_cleaned = cleaned_dataset['clean_comment']
y_cleaned = cleaned_dataset['category']

# Split the cleaned data into train and test sets (80-20 split)
X_train_cleaned, X_test_cleaned, y_train_cleaned, y_test_cleaned = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42)

# Apply TfidfVectorizer with trigram setting and max_features=10000
tfidf_cleaned = TfidfVectorizer(ngram_range=(1, 3), max_features=10000)

# Fit the vectorizer on the training data and transform both train and test sets
X_train_tfidf_cleaned = tfidf_cleaned.fit_transform(X_train_cleaned)
X_test_tfidf_cleaned = tfidf_cleaned.transform(X_test_cleaned)

# Base learners
lightgbm_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    metric="multi_logloss",
    is_unbalance=True,
    class_weight="balanced",
    reg_alpha=0.1,  # L1 regularization
    reg_lambda=0.1,  # L2 regularization,
    learning_rate=0.08081298097796712,
    n_estimators=367,
    max_depth=20
)

logreg_model = LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs', multi_class='multinomial')

# Meta-learner
knn_meta_learner = KNeighborsClassifier(n_neighbors=5)

# Create the StackingClassifier with LightGBM and LogisticRegression as base models, and KNN as meta-learner
stacking_model = StackingClassifier(
    estimators=[
        ('lightgbm', lightgbm_model),
        ('logistic_regression', logreg_model)
    ],
    final_estimator=knn_meta_learner,
    cv=5
)

# Train the stacking model
stacking_model.fit(X_train_tfidf_cleaned, y_train_cleaned)

# Make predictions on the test data
y_pred = stacking_model.predict(X_test_tfidf_cleaned)

# Generate classification report
print(classification_report(y_test_cleaned, y_pred))